# Stoa 03 — preprocess into the two final datasets

Reads what **`02_combine.ipynb`** saved and writes the two datasets you work with, to `data/final/`. Nothing is re-downloaded, and combine's outputs are not changed.

| Final dataset | One row per | Key | What's in it |
|---|---|---|---|
| `data/final/sensor_steepness.parquet` | sensor (118) | `location_id` | location, `slope_pct`, `elevation_m`, direction labels, how each value was matched |
| `data/final/pcs_hourly.parquet` | sensor-hour (~7.4M) | `location_id` + `date` + `time` | lat/long, pedestrian count, direction counts, outage flag, weather |

Link them by `location_id` when you need slope or elevation next to the counts (see the end of this notebook).

| Input (from combine) | Used for |
|---|---|
| `data/processed/sensor_table.parquet` | the steepness dataset |
| `data/processed/pcs_hourly.parquet` | lat/long + counts + weather per sensor-hour |
| `data/interim/counts_clean.parquet` | the direction counts (combine left them out of its hourly table) |
| `data/raw/pcs_sensor_locations*.{parquet,json}` | what each direction means (North/South, East/West) |
| `data/raw/ga_dem_5m_samples.json` | LiDAR ground heights, for sensors with no footpath nearby |

**What this notebook does to combine's tables**

Cleaning and gap-filling:
1. **Indoor sensors dropped** from both datasets.
2. **LiDAR fallback:** sensors with no footpath within 25 m (combine left their slope/elevation empty) get them from the LiDAR heights.
3. **Outage zeros flagged** (`suspect_outage_zero`).
4. **Ends at the last hour with weather.** ERA5 runs about 6 days behind, so the newest count hours have no weather yet; they're dropped so every row has all its columns. The end date moves forward with each new weather download.

Layout:
5. **Time is UTC only, as two columns.** Combine already converted Melbourne clock time to UTC correctly (daylight saving included), so the local columns (`timestamp_local`, `sensing_date`, `hourday`) are dropped. `timestamp_utc` is split into `date` (UTC date) and `time` (UTC time of day, e.g. `14:00:00`; counts are hourly, so it's always on the hour).
6. **Direction counts added back** (`direction_1`, `direction_2`), for now. They exist only from July 2021; every earlier row (the whole archive) is `NaN`.
7. **Direction labels** added to the steepness dataset.

## 1. Setup

In [ ]:
%pip install -q pyproj

In [ ]:
import os
import sys
from pathlib import Path

# Where the data lives. Colab: your Google Drive. A local clone of the repo: data/notebooks/,
# kept apart from the team pipeline's data/raw, data/interim and data/processed (git-ignored).
# Set STOA_NOTEBOOK_DATA to use another folder.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/stoa/data")
else:
    DATA = Path(os.environ.get("STOA_NOTEBOOK_DATA", Path.cwd().resolve().parent / "data" / "notebooks"))
DATA.mkdir(parents=True, exist_ok=True)
print("data folder:", DATA)

In [ ]:
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pyproj import Transformer

RAW = DATA / "raw"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"   # combine's outputs (read only)
FINAL = DATA / "final"           # the two final datasets
FINAL.mkdir(parents=True, exist_ok=True)

# LiDAR fallback: slope = height difference across +/-10 m around the sensor.
DEM_STEP_M = 10.0
DEM_CACHE = RAW / "ga_dem_5m_samples.json"   # saved by 01_download (source "ga_dem")

# A sensor reading exactly 0 for at least this many hours in a row is treated as a suspected outage.
OUTAGE_MIN_HOURS = 24

qa: dict = {}   # counts and checks, saved as final/qa_report.json

## 2. Steepness dataset (one row per sensor)

Combine's sensor table, in three steps: drop indoor sensors, fill slope/elevation from LiDAR where there's no footpath nearby, and add what each direction count means.

### 2a. Indoor sensors

The current sensor table marks 34 sensors as `Indoor` (e.g. 80-106, 144-160). Pedestrian counts indoors don't depend on the street, slope or weather, so they're dropped. None of them appear in any of the three count files today, so this removes nothing (`sensors_dropped_indoor = []`); it stays as a safeguard in case a future live download includes them. (The original Colab notebook removed them from its sensor list because it started from all 134 sensors; this pipeline starts from the sensors that have counts.)

The 18 retired sensors come from the 2022 snapshot, which has no indoor/outdoor column, so their `location_type` is empty; by name they're street sensors.

In the raw sensor tables, `direction_1` / `direction_2` are **labels** (e.g. "North" / "South"), saying which way each direction count goes. They're renamed `direction_1_label` / `direction_2_label` so they can't be confused with the **count** columns of the same name in the hourly dataset.

Labels come from the current sensor table; the 18 retired sensors fall back to the Oct 2022 snapshot.

**Caution:** for the 31 North/South sensors in both tables, the 2022 snapshot says direction 1 = **South** and the current table says **North**. East/West sensors agree. So it isn't certain which way older North/South rows counted, and the 10 retired North/South sensors (2022 labels) may use the opposite convention to the current ones.

In [ ]:
sensors = pd.read_parquet(PROCESSED / "sensor_table.parquet")

indoor = sensors.location_type.eq("Indoor")
qa["sensors_dropped_indoor"] = sensors.location_id[indoor].tolist()
sensors = sensors[~indoor].reset_index(drop=True)
print(f"{len(sensors)} sensors; indoor sensors dropped: {qa['sensors_dropped_indoor']}")

### 2b. LiDAR fallback for sensors with no footpath nearby

Combine left `slope_pct` and `elevation_m` empty for sensors more than 25 m from any footpath section (`slope_matched = False`; sensors 25 and 130 in the last run), because the nearest section is on a different street.

Here they come from Geoscience Australia's 5 m LiDAR elevation model (a laser survey flown over the city), using the heights `01_download.ipynb` saved at the sensor and 10 m east, west, north and south:
- **elevation** = the height at the sensor.
- **slope** = how much the height changes across the 20 m between opposite points, combining east-west and north-south, as a percentage (same unit as `gradepc`: metres up per 100 m).

`place_source` says where each sensor's values came from.

In [ ]:
def dem_height(lon: float, lat: float, cache: dict) -> float:
    # Look up the ground height at one point in the heights 01_download saved.
    key = f"{lon:.7f},{lat:.7f}"
    if key not in cache:
        raise KeyError(f"no LiDAR height saved for {key}; re-run the 'ga_dem' source in 01_download.ipynb")
    if cache[key] is None:   # the model has no ground there (e.g. over water)
        raise ValueError(f"LiDAR has no height at {key}; this sensor needs another source of elevation")
    return cache[key]


far = ~sensors.slope_matched
if far.any():
    to_wgs = Transformer.from_crs("EPSG:7855", "EPSG:4326", always_xy=True)   # metres -> lat/long
    cache = json.loads(DEM_CACHE.read_text())
    d = DEM_STEP_M
    for i in sensors.index[far]:
        x, y = sensors.at[i, "easting_m"], sensors.at[i, "northing_m"]
        z = {k: dem_height(*to_wgs.transform(x + dx, y + dy), cache)
             for k, dx, dy in [("c", 0, 0), ("e", d, 0), ("w", -d, 0), ("n", 0, d), ("s", 0, -d)]}
        slope = float(np.hypot(z["e"] - z["w"], z["n"] - z["s"]) / (2 * d) * 100)
        sensors.loc[i, ["elevation_m", "slope_pct"]] = [z["c"], slope]
sensors.loc[far, "place_source"] = "ga_dem_5m"
qa["sensors_filled_from_ga_dem"] = sensors.location_id[far].tolist()

sensors.loc[far, ["location_id", "sensor_name", "match_distance_m", "slope_pct", "elevation_m", "place_source"]]

### 2c. Direction labels, then save

In [ ]:
cur = pd.read_parquet(RAW / "pcs_sensor_locations.parquet", columns=["location_id", "direction_1", "direction_2"])
old = pd.DataFrame(json.loads((RAW / "pcs_sensor_locations_2022-10-24_wayback.json").read_text("utf-8")))
old = pd.DataFrame({"location_id": old.sensor_id.astype(int), "direction_1": old.direction_1, "direction_2": old.direction_2})

# Current table first; the 2022 snapshot only for sensors the current table doesn't have.
labels = pd.concat([cur, old[~old.location_id.isin(cur.location_id)]], ignore_index=True)
labels = labels.rename(columns={"direction_1": "direction_1_label", "direction_2": "direction_2_label"})
labels["location_id"] = labels.location_id.astype("int16")

sensors = sensors.merge(labels, on="location_id", how="left", validate="one_to_one")
qa["sensors"] = len(sensors)
qa["sensors_with_direction_labels"] = int(sensors.direction_1_label.notna().sum())
sensors.to_parquet(FINAL / "sensor_steepness.parquet", index=False)

sensors[["location_id", "sensor_name", "latitude", "longitude", "direction_1_label", "direction_2_label",
         "slope_pct", "elevation_m"]].head(10)

## 3. Hourly dataset (one row per sensor-hour)

1. Start from combine's hourly table (lat/long + counts + weather), keeping only the sensors in the steepness dataset (this drops indoor sensors, if any).
2. **End at the last hour with weather** (ERA5 lag, ~6 days before the latest count).
3. **Flag outage zeros** (see below).
4. Add the direction counts from `counts_clean`, matched on the exact sensor and UTC hour. Combine's hourly table was built from the same cleaned counts, so every row has exactly one match.
5. If the hourly table has no latitude / longitude (made by an older combine), add them from the steepness dataset by `location_id`.
6. Drop the local-time columns and split `timestamp_utc` into `date` + `time` (`split_utc` below).
7. Store the direction counts as decimals so rows without directions show `NaN` (whole-number columns can't hold `NaN`). Counts stay exact: a float holds every whole number this size.

Each merge is checked so that it can't add or lose rows.

**Outage zeros.** Sometimes a sensor breaks or loses power but keeps reporting **0 people** every hour instead of nothing:

```
Swanston St sensor:  …  812  640  0  0  0  0  0  … (0 for 3 days straight) …  0  0  755  901 …
                                  └──────── broken, not an empty street ────────┘
```

A busy city street is almost never at exactly 0 for a whole day, so runs of **24+ consecutive hours of 0** at one sensor are flagged `suspect_outage_zero = True`. They're not deleted: decide later whether to treat them as missing (a model would otherwise learn "sometimes this street is empty all day").

In [ ]:
def split_utc(df: pd.DataFrame) -> pd.DataFrame:
    # Put date and time of day (both UTC) where timestamp_utc was, then drop the timestamp.
    df = df.copy()
    pos = df.columns.get_loc("timestamp_utc")
    df.insert(pos, "date", df.timestamp_utc.dt.date)
    df.insert(pos + 1, "time", df.timestamp_utc.dt.time)
    return df.drop(columns="timestamp_utc")


hourly = pd.read_parquet(PROCESSED / "pcs_hourly.parquet")
n_combine = len(hourly)

# Only sensors in the steepness dataset (drops indoor sensors, if any).
hourly = hourly[hourly.location_id.isin(sensors.location_id)]
qa["rows_dropped_indoor"] = n_combine - len(hourly)

# End at the last hour with weather. ERA5 runs ~6 days behind the counts, so the newest hours
# have no weather; cutting there (rather than a fixed number of days) keeps every hour that has it.
WEATHER = ["temp_c", "wind_kmh", "rain_mm"]   # is_day is calculated, so it's never missing
last_weather = hourly.timestamp_utc[hourly[WEATHER].notna().all(axis=1)].max()
qa["rows_dropped_after_last_weather"] = int((hourly.timestamp_utc > last_weather).sum())
qa["last_count_hour_utc"] = str(hourly.timestamp_utc.max())
qa["end_utc"] = str(last_weather)
hourly = hourly[hourly.timestamp_utc <= last_weather]
print(f"ends at {last_weather} UTC (last hour with weather); dropped {qa['rows_dropped_after_last_weather']:,} "
      f"newer rows (counts ran to {qa['last_count_hour_utc']})")
n_in = len(hourly)

# Outage zeros: runs of consecutive 0-count hours at one sensor. A new run starts whenever the
# zero/non-zero state changes, the sensor changes, or there's a gap in time (a missing hour).
hourly = hourly.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
z = hourly.pedestriancount.eq(0)
step = hourly.timestamp_utc.diff().ne(pd.Timedelta(hours=1))
run = (z.ne(z.shift()) | hourly.location_id.ne(hourly.location_id.shift()) | step).cumsum()
hourly["suspect_outage_zero"] = z & (run.map(run.value_counts()) >= OUTAGE_MIN_HOURS)
qa["rows_flagged_suspect_outage_zero"] = int(hourly.suspect_outage_zero.sum())

directions = pd.read_parquet(INTERIM / "counts_clean.parquet",
                             columns=["location_id", "timestamp_utc", "direction_1", "direction_2"])
hourly = hourly.merge(directions, on=["location_id", "timestamp_utc"], how="left", validate="one_to_one")

if "latitude" not in hourly.columns:   # older combine output without lat/long
    place = sensors[["location_id", "latitude", "longitude"]]
    hourly = hourly.merge(place, on="location_id", how="left", validate="many_to_one")
assert len(hourly) == n_in, "a merge changed the row count"

# Local time was only needed to work out UTC, which combine already did.
hourly = hourly.drop(columns=["timestamp_local", "sensing_date", "hourday"])
hourly = split_utc(hourly.sort_values(["timestamp_utc", "location_id"]))

# No direction data (all archive rows, i.e. before July 2021) -> NaN.
hourly[["direction_1", "direction_2"]] = hourly[["direction_1", "direction_2"]].astype("float64")

# Final column order: who/where, when, target, directions, flags, weather.
first = ["location_id", "latitude", "longitude", "date", "time",
         "pedestriancount", "direction_1", "direction_2", "source", "suspect_outage_zero"]
hourly = hourly[first + [c for c in hourly.columns if c not in first]]
hourly.to_parquet(FINAL / "pcs_hourly.parquet", index=False, compression="zstd")
hourly.head()

## 4. Checks

In [ ]:
has_dir = hourly.direction_1.notna() & hourly.direction_2.notna()
checks = {
    "row_count_unchanged": len(hourly) == n_in,   # after dropping indoor sensors
    "no_indoor_sensors": not sensors.location_type.eq("Indoor").any(),
    "every_sensor_has_slope_and_elevation": bool(sensors.slope_pct.notna().all() and sensors.elevation_m.notna().all()),
    "outage_flag_only_on_zero_counts": not (hourly.suspect_outage_zero & hourly.pedestriancount.ne(0)).any(),
    "no_duplicate_sensor_hours": not hourly.duplicated(["location_id", "date", "time"]).any(),
    "every_row_has_lat_long": bool(hourly.latitude.notna().all() and hourly.longitude.notna().all()),
    "every_row_has_weather": bool(hourly[WEATHER + ["is_day"]].notna().all().all()),
    "every_time_on_the_hour": bool(hourly.time.drop_duplicates().map(lambda t: t.minute == 0 and t.second == 0).all()),
    "direction_sum_matches_total": bool(
        (hourly.direction_1[has_dir] + hourly.direction_2[has_dir] == hourly.pedestriancount[has_dir]).all()),
    # Directions only exist in the Wayback and live sources (July 2021 onward).
    "no_directions_in_archive_rows": not has_dir[hourly.source.eq("archive")].any(),
    # The two datasets must link cleanly: one row per sensor, and every hourly sensor has one.
    "steepness_one_row_per_sensor": not sensors.location_id.duplicated().any(),
    "every_hourly_sensor_in_steepness": bool(hourly.location_id.isin(sensors.location_id).all()),
}
qa["rows"] = len(hourly)
qa["rows_with_directions"] = int(has_dir.sum())
qa["first_date_with_directions"] = str(hourly.date[has_dir].min())
qa["range_utc"] = [str(hourly.date.min()), str(hourly.date.max())]
qa["checks"] = checks
(FINAL / "qa_report.json").write_text(json.dumps(qa, indent=2, default=str))
print(json.dumps(qa, indent=2, default=str))

failed = [k for k, ok in checks.items() if not ok]
if failed:
    raise RuntimeError(f"checks failed: {failed}")

## 5. Size

In [ ]:
for p in sorted(FINAL.glob("*.parquet")):
    meta = pq.ParquetFile(p).metadata
    print(f"{p.name:<26} {meta.num_rows:>11,} rows  {meta.num_columns:>3} columns  {p.stat().st_size / 1e6:7.1f} MB on disk")
print(f"\nhourly dataset in memory (pandas): {hourly.memory_usage(deep=True).sum() / 1e9:.2f} GB")

## 6. Using the two datasets together

Add slope and elevation to the hourly rows only when you need them, by `location_id`. `validate="many_to_one"` stops if a sensor somehow appears twice in the steepness dataset.

In [ ]:
steep = pd.read_parquet(FINAL / "sensor_steepness.parquet", columns=["location_id", "slope_pct", "elevation_m"])
example = hourly.tail(1000).merge(steep, on="location_id", how="left", validate="many_to_one")
example.head()